In [1]:
!pip install transformers datasets -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.6/193.6 kB 7.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.8.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
cesium 0.12.4 requires numpy<3.0,>=2.0, but you have numpy 1.26.4 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cublas-cu12==12.4.5.8; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cublas-cu12 12.5.3.2 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cuda-cupti-cu12==12.4.127; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cuda-cupti-cu12 12.5.82 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cuda-nvrtc-cu12==12.4.127; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cuda-nvrtc-cu12 12.5.82 which is incompatible.
t

In [3]:
# =========================================
# Prerequisites:
# =========================================
# pip install pandas torch transformers tqdm scikit-learn
# GPU recommended for faster training
# =========================================

import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from torch.optim import AdamW
from tqdm.auto import tqdm
import os
from sklearn.metrics import accuracy_score

# ============================
# 1. Load train/val/test CSVs
# ============================
train_df = pd.read_csv("/kaggle/input/final-dataset2/train.csv")
val_df   = pd.read_csv("/kaggle/input/final-dataset2/validation.csv")
test_df  = pd.read_csv("/kaggle/input/final-dataset2/test.csv")

# Extract text and labels
train_texts = train_df["clean_sentence"].tolist()
train_labels = train_df["intensity_level_normalized"].astype(int).tolist()

val_texts = val_df["clean_sentence"].tolist()
val_labels = val_df["intensity_level_normalized"].astype(int).tolist()

test_texts = test_df["clean_sentence"].tolist()
test_labels = test_df["intensity_level_normalized"].astype(int).tolist()

num_labels = len(set(train_labels + val_labels + test_labels))

# ============================
# 2. Dataset Class
# ============================
model_name = "bert-base-multilingual-cased"  # mBERT
tokenizer = AutoTokenizer.from_pretrained(model_name)

class BanglaDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = self.labels[idx]
        encoding = self.tokenizer(
            text,
            padding='max_length',
            truncation=True,
            max_length=self.max_len,
            return_tensors='pt'
        )
        return {
            'input_ids': encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
            'labels': torch.tensor(label, dtype=torch.long)
        }

train_dataset = BanglaDataset(train_texts, train_labels, tokenizer)
val_dataset   = BanglaDataset(val_texts, val_labels, tokenizer)
test_dataset  = BanglaDataset(test_texts, test_labels, tokenizer)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=32, shuffle=False)
test_loader  = DataLoader(test_dataset, batch_size=32, shuffle=False)

# ============================
# 3. Model + Optimizer
# ============================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=num_labels)
model.to(device)

optimizer = AdamW(model.parameters(), lr=2e-5)

# ============================
# 4. Training Loop with Validation Accuracy
# ============================
epochs = 5
save_dir = "/kaggle/working/checkpoints_mbert"
os.makedirs(save_dir, exist_ok=True)

for epoch in range(epochs):
    # ---- Training ----
    model.train()
    loop = tqdm(train_loader, leave=True)
    total_loss = 0
    for batch in loop:
        optimizer.zero_grad()
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        loop.set_description(f"Epoch {epoch+1}/{epochs}")
        loop.set_postfix(loss=loss.item())

    avg_loss = total_loss / len(train_loader)

    # ---- Validation Accuracy ----
    model.eval()
    all_preds = []
    all_labels = []
    with torch.no_grad():
        for batch in val_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            preds = torch.argmax(outputs.logits, dim=1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    val_acc = accuracy_score(all_labels, all_preds)
    print(f"Epoch {epoch+1} finished. Average loss: {avg_loss:.4f} | Val Accuracy: {val_acc*100:.2f}%")

    # ---- Save model after each epoch ----
    epoch_save_path = os.path.join(save_dir, f"model_epoch_{epoch+1}")
    model.save_pretrained(epoch_save_path)
    tokenizer.save_pretrained(epoch_save_path)
    print(f"Saved checkpoint: {epoch_save_path}")

print("✅ Training done. All checkpoints saved in /kaggle/working/checkpoints_mbert")

# ============================
# 5. Final Test Accuracy
# ============================
model.eval()
test_preds = []
test_gt = []
with torch.no_grad():
    for batch in test_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        preds = torch.argmax(outputs.logits, dim=1)
        test_preds.extend(preds.cpu().numpy())
        test_gt.extend(labels.cpu().numpy())

test_acc = accuracy_score(test_gt, test_preds)
print(f"🔹 Final Test Accuracy: {test_acc*100:.2f}%")


tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/996k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.96M [00:00<?, ?B/s]

2025-09-28 16:12:36.266090: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1759075956.419667      36 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1759075956.474370      36 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


...

Epoch 3 finished. Average loss: 0.7635 | Val Accuracy: 71.40%
Saved checkpoint: /kaggle/working/checkpoints_mbert/model_epoch_3


  0%|          | 0/471 [00:00<?, ?it/s]

Epoch 4 finished. Average loss: 0.6069 | Val Accuracy: 73.26%
Saved checkpoint: /kaggle/working/checkpoints_mbert/model_epoch_4


  0%|          | 0/471 [00:00<?, ?it/s]

Epoch 5 finished. Average loss: 0.4503 | Val Accuracy: 71.96%
Saved checkpoint: /kaggle/working/checkpoints_mbert/model_epoch_5
✅ Training done. All checkpoints saved in /kaggle/working/checkpoints_mbert
🔹 Final Test Accuracy: 71.75%
